**Goal: explain choice of data and project goal**

I chose the following dataset: 

@ *https://huggingface.co/datasets/ccdv/arxiv-summarization*

It alligned with what I want to accomplish which is title generation for academic papers. The dataset contains papers from multiple fields of STEM allowing me to focus on either one field or multiple for my model. It also helps that its pretty organized

**Project data summary: data index, format, size in GB, # rows, statistical characteristics of data**

Formatting: *Apache Parquet*

Size: *7.26 GB*

Number of Rows: *431,826*

Characteristics: *Word Counts, Character Counts*

In [1]:
import pandas as pd
from datasets import load_dataset

c:\Users\Widi\anaconda3\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ds = load_dataset("ccdv/arxiv-summarization", "section")

c:\Users\widin\anaconda3\envs\nlp\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\widin\.cache\huggingface\hub\datasets--ccdv--arxiv-summarization. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 6440/6440 [00:00<00:00, 8394.77 examples/s] 


In [5]:
print(ds)
print(ds["train"].column_names)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 203037
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6436
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6440
    })
})
['article', 'abstract']
{'article': 'additive models @xcite provide an important family of models for semiparametric regression or classification . some reasons for the success of additive models are their increased flexibility when compared to linear or generalized linear models and their increased interpretability when compared to fully nonparametric models . \n it is well - known that good estimators in additive models are in general less prone to the curse of high dimensionality than good estimators in fully nonparametric models . \n many examples of such estimators belong to the large class of regularized kernel based methods over a reproducing kernel hilbert space @xmath0 , see e.

In [6]:
dataset_doc = load_dataset("ccdv/arxiv-summarization", "document")

Generating test split: 100%|██████████| 6440/6440 [00:00<00:00, 9543.23 examples/s] 


In [7]:
print(dataset_doc["train"].column_names)
print(dataset_doc["train"][0].keys())

['article', 'abstract']
dict_keys(['article', 'abstract'])


This dataset has plentiful to it but only contains the abstract and article. No title and no id to trace back the article to get a title. Another concern that presented itself after viewing this dataset is that if I decide to train on both the abstract portion and the article of a paper, it will take a lot of computational power that I don't have. So I decided to pivot over and started looking for datasets that contained at least the title of the paper and the abstract.


I found the following:

@ *https://huggingface.co/datasets/librarian-bots/arxiv-metadata-snapshot*

In [2]:
dataset = load_dataset("librarian-bots/arxiv-metadata-snapshot")

In [3]:
print(dataset["train"].column_names)
print(dataset["train"][0])

['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed']
{'id': '0809.2509', 'submitter': 'Vitaly Velizhanin', 'authors': 'V.N. Velizhanin (St. Petersburg, INP)', 'title': 'Three-loop renormalization of the N=1, N=2, N=4 supersymmetric Yang-Mills theories', 'comments': '6 pages, mismatch for N=2 SYM theory corrected, references updated', 'journal-ref': None, 'doi': '10.1016/j.nuclphysb.2009.03.017', 'report-no': None, 'categories': 'hep-th', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'abstract': 'We calculate the renormalization constants of the N=1, N=2, N=4 supersymmetric Yang-Mills theories in an arbitrary covariant gauge in the dimensional reduction scheme up to three loops. We have found, that the beta-functions for N=1 and N=4 SYM theories are the same from the different triple vertices. This means that the dimensional reduction scheme works corre

It seems the second one is more up to date and has more content to it. Also as another plus it has a categories column allowing me to pick and choose on training CS papers for CS title predictions or Bio for Bio titles. So essentially try small subsets of the dataset, see how it works and if its going well then I'll scale from domain-specific to general STEM papers (start from CS, Math, Physics).

So I am going to pivot over to this dataset instead

**New Dataset:**

Formatting: *Apache Parquet*

Size: *2.77 GB*

Number of Rows: *2,975,294*

Characteristics: *id, submitter, authors, title, comments, journal-ref, doi, report-no, categories, license, abstract, versions, update_date, authors_parsed*

Not only does this dataset have what im looking for (title, abstract), it also contains a bunch of other characteristics, more notably the id. This will allow me to trace back the original document and scrape for more information if I decide to test with more than just the abstract portions. But for now I'm sticking with the title and abstract.

**Data Ingestion**

I don't need all of the columns so we can end up dropping most of them

In [5]:
dataset = dataset["train"].select_columns(["id","title", "abstract", "categories"])

In [7]:
df = dataset.to_pandas()
print(f"Columns: {list(df.columns)}")
print(f"Length: {len(df)}")

Columns: ['id', 'title', 'abstract', 'categories']
Length: 2975294


And just to see how much we actually took out:

In [8]:
print(f"Sample: {df.iloc[0]}")

Sample: id                                                    0809.2509
title         Three-loop renormalization of the N=1, N=2, N=...
abstract      We calculate the renormalization constants of ...
categories                                               hep-th
Name: 0, dtype: str


When I was looking at the dataset on HuggingFace I noticed some symbols that are native to LaTeX and I don't need them because there aren't any symbols in the titles of these papers so I will clean them out

In [ ]:
import re

In [11]:
def clean(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\\[a-zA-Z]+\{([^}]*)\}", r"\1", text) #Takes out any LaTeX commands
    text = re.sub(r"\\[a-zA-Z]+", "", text) #More LaTeX commands
    text = re.sub(r"\$([^$]*)\$", r"\1", text) #Takes out $ math symbols
    text = re.sub(r'\s+', ' ', text) #Takes care of whitespaces
    
    return text.strip()

In [12]:
df["title"] = df["title"].apply(clean)
df["abstract"] = df["abstract"].apply(clean)

In [13]:
#Takes care of any rows with an empty title or an empty abstract (as a precautionary measure)
df = df[(df["title"].str.len() > 0) & (df["abstract"].str.len() > 0)]

This is how its looking so far:

In [14]:
print(len(df))

2975294


The # of original rows is still the same (2,975,294) so there weren't any missing rows

As I mention earlier, I want to start with CS and Math papers first and the categories column are organized using cs.<> or math.<>. I don't know how much subsets of each category there are so the following should tell me:

In [16]:
domains = {
    "cs":"cs.",
    "math": "math.",
}

def has_category_prefix(categories_string, prefix):
    return any(cat.startswith(prefix) for cat in categories_string.split())

for domain_name, prefix in domains.items():
    subset = df[df["categories"].apply(lambda x: has_category_prefix(x, prefix))]
    print(f"{domain_name}: {len(subset)} papers")

cs: 889249 papers
math: 741037 papers


For further details of the data:

In [17]:
df["title_length"] = df["title"].str.split().apply(len)
df["abstract_length"] = df["abstract"].str.split().apply(len)

print(df["title_length"].describe())
print(df["abstract_length"].describe())

count    2.975294e+06
mean     9.830362e+00
std      3.673751e+00
min      1.000000e+00
25%      7.000000e+00
50%      9.000000e+00
75%      1.200000e+01
max      6.100000e+01
Name: title_length, dtype: float64
count    2.975294e+06
mean     1.444452e+02
std      6.278623e+01
min      1.000000e+00
25%      9.700000e+01
50%      1.410000e+02
75%      1.890000e+02
max      9.250000e+02
Name: abstract_length, dtype: float64


In [23]:
cs_subset = df[df["categories"].apply(lambda x: has_category_prefix(x, "cs."))]
math_subset = df[df["categories"].apply(lambda x: has_category_prefix(x, "math."))]

print(cs_subset.head())
print(math_subset.head())


            id                                              title  \
7   1906.03451  The probabilistic superiority of stochastic sy...   
10  1911.06442      Monotone Comparative Statics without Lattices   
12  2006.01357  Large deviations principles for symplectic dis...   
15  2102.04061  Convergence analysis for minimum action method...   
16  2112.13243  Motion Illusions Generated Using Predictive Ne...   

                                             abstract             categories  \
7   It is well known that symplectic methods have ...          math.NA cs.NA   
10  The theory of Monotone Comparative Statics (MC...          econ.TH cs.GT   
12  In this paper, we consider the large deviation...          math.NA cs.NA   
15  The minimum action method (MAM) is an effectiv...  math.PR cs.NA math.NA   
16  Why do we sometimes perceive static images as ...      cs.NE cs.AI cs.CV   

    title_length  abstract_length  
7             11              226  
10             5               8

Our Final dataframe:

In [19]:
df

,id,title,abstract,categories,title_length,abstract_length
0,0809.2509,"Three-loop renormalization of the N=1, N=2, N=...",We calculate the renormalization constants of ...,hep-th,10,66
1,1302.0173,Basic aspects of high-power semiconductor lase...,The aim of this paper is to review some of the...,physics.optics,7,125
2,1503.06703,The *-variation of the Banach-Mazur game and f...,We introduce a property of posets which streng...,math.LO,9,113
3,1511.07051,"Firewalls, black-hole thermodynamics, and sing...",We investigate thermodynamic equilibrium of a ...,gr-qc hep-th quant-ph,10,117
4,1810.05608,Limits of conformal images and conformal image...,We address the scaling limits of random curves...,math-ph math.MP math.PR,13,143
...,...,...,...,...,...,...
2975289,solv-int/9912005,"Generalized KP hierarchy: M\""obius Symmetry, S...",Analytic-bilinear approach is used to study co...,solv-int nlin.SI,10,77
2975290,solv-int/9912008,"The Pfaff lattice, Matrix integrals and a map ...","We study the Pfaff lattice, introduced by us i...",solv-int nlin.SI,12,150
2975291,solv-int/9912011,Liouville equation under perturbation,Small perturbation of the Liouville equation u...,solv-int nlin.SI,4,29
2975292,solv-int/9912013,Singular solution of the Liouville equation un...,Small perturbation of the Liouville equation u...,solv-int nlin.SI,8,41


**Data Normalizaion:** *Sandbox*

Sentence Tokenizers

Word Tokenizers

Token Normalize

From what I've seen in other projects and online discussion, I settled on some of the tokenizers used and mentioned

In [24]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize, TreebankWordTokenizer, TweetTokenizer

Due to the sheer size of my dataset, I'm going to stick with just a portion for demo

In [25]:
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

SAMPLE_SIZE = 50000
RANDOM_SEED = 67

df_sample = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)

*Sentence Tokenization:*

In [50]:
import time
from collections import Counter

In [52]:
df_sample["sentences"] = df_sample["abstract"].apply(sent_tokenize)
df_sample["num_sentences"] = df_sample["sentences"].apply(len)

print(f"\nSentences per abstract:")
print(df_sample["num_sentences"].describe())


Sentences per abstract:
count    50000.000000
mean         6.113880
std          2.760316
min          1.000000
25%          4.000000
50%          6.000000
75%          8.000000
max         27.000000
Name: num_sentences, dtype: float64


*Word Tokenization:*

So I settled on the following word tokenizers to test head to head:

*word_tokenize*

*TreebankWordTokenizer*

*TweetTokenizer*

I'm confident about the first choice. I threw in the *TweetTokenizer* as a just cause since I'm dealing with academic papers and *TweetTokenizer* specializes in social media postings. It would just be interesting to see the contrast. And *TreebankWordTokenizer* will act as a baseline.

In [54]:
sample_abstract = df_sample["abstract"].iloc[0]
print(f"First 300 characters: \n{sample_abstract[:300]}...\n")

First 300 characters: 
Accurate forecasting of electrical demand is essential for maintaining a stable and reliable power grid, optimizing the allocation of energy resources, and promoting efficient energy consumption practices. This study investigates the effectiveness of five hyperparameter optimization (HPO) algorithms...



In [55]:
tokenizers = {
    "word_tokenize": lambda text: word_tokenize(text),
    "TreebankWordTokenizer": lambda text: TreebankWordTokenizer().tokenize(text),
    "TweetTokenizer": lambda text: TweetTokenizer().tokenize(text),
}

#first 20 tokens of a sample abstract:

for name, tok_fn in tokenizers.items():
    tokens = tok_fn(sample_abstract)
    print(f"\n{name}:")
    print(f"  Tokens: {tokens[:20]}")
    print(f"  Total token count: {len(tokens)}")



word_tokenize:
  Tokens: ['Accurate', 'forecasting', 'of', 'electrical', 'demand', 'is', 'essential', 'for', 'maintaining', 'a', 'stable', 'and', 'reliable', 'power', 'grid', ',', 'optimizing', 'the', 'allocation', 'of']
  Total token count: 197

TreebankWordTokenizer:
  Tokens: ['Accurate', 'forecasting', 'of', 'electrical', 'demand', 'is', 'essential', 'for', 'maintaining', 'a', 'stable', 'and', 'reliable', 'power', 'grid', ',', 'optimizing', 'the', 'allocation', 'of']
  Total token count: 191

TweetTokenizer:
  Tokens: ['Accurate', 'forecasting', 'of', 'electrical', 'demand', 'is', 'essential', 'for', 'maintaining', 'a', 'stable', 'and', 'reliable', 'power', 'grid', ',', 'optimizing', 'the', 'allocation', 'of']
  Total token count: 201


In [56]:
#across full sample:

tokenizer_results = {}
for name, tok_fn in tokenizers.items():
    start = time.time()
    token_counts = df_sample["abstract"].apply(lambda x: len(tok_fn(x)))
    elapsed = time.time() - start

    tokenizer_results[name] = {
        "time": elapsed,
        "mean_tokens": token_counts.mean(),
        "median_tokens": token_counts.median(),
        "std_tokens": token_counts.std(),
    }

    print(f"\n{name}:")
    print(f"Time: {elapsed:.2f}s for {SAMPLE_SIZE:,} abstracts")
    print(f"Mean tokens/abstract: {token_counts.mean():.1f}")
    print(f"Median tokens/abstract: {token_counts.median():.1f}")
    print(f"Std: {token_counts.std():.1f}")


word_tokenize:
Time: 17.22s for 50,000 abstracts
Mean tokens/abstract: 164.5
Median tokens/abstract: 160.0
Std: 72.6

TreebankWordTokenizer:
Time: 10.33s for 50,000 abstracts
Mean tokens/abstract: 159.5
Median tokens/abstract: 155.0
Std: 70.4

TweetTokenizer:
Time: 20.48s for 50,000 abstracts
Mean tokens/abstract: 167.4
Median tokens/abstract: 162.0
Std: 74.4


*Token Normalization:*

In [57]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [58]:
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [59]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def normalize_tokens(text):
    tokens = word_tokenize(text)
    tokens = [t.lower() for t in tokens]
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

df_sample["normalized_tokens"] = df_sample["abstract"].apply(normalize_tokens)
df_sample["normalized_len"] = df_sample["normalized_tokens"].apply(len)


In [60]:
#before vs after
raw_counts = df_sample["abstract"].apply(lambda x: len(word_tokenize(x)))
print(f"Before: {raw_counts.mean():.1f}")
print(f"After: {df_sample['normalized_len'].mean():.1f}")
print(f"Token reduction: {(1 - df_sample['normalized_len'].mean() / raw_counts.mean()) * 100:.2f}%")

Before: 164.5
After: 82.0
Token reduction: 50.16%


In [61]:
all_raw_tokens = df_sample["abstract"].apply(word_tokenize).explode()
all_norm_tokens = df_sample["normalized_tokens"].explode()
print(f"Size raw: {all_raw_tokens.nunique():,}")
print(f"Size normalized: {all_norm_tokens.nunique():,}")
print(f"Vocabulary reduction: {(1 - all_norm_tokens.nunique() / all_raw_tokens.nunique()) * 100:.2f}%")

Size raw: 190,968
Size normalized: 65,744
Vocabulary reduction: 65.57%


And just to see the most common tokens:

In [63]:
top_tokens = Counter(all_norm_tokens).most_common(20)
for token, count in top_tokens:
    print(f"  {token}: {count:,}")

  model: 39,128
  result: 21,681
  method: 20,745
  system: 20,432
  data: 19,008
  show: 18,096
  using: 16,374
  study: 14,628
  also: 14,619
  two: 13,692
  field: 13,481
  paper: 12,831
  approach: 12,657
  state: 12,571
  problem: 12,461
  function: 11,847
  time: 11,699
  quantum: 10,808
  network: 10,793
  one: 10,662
